In [ ]:
import joblib
import pandas as pd

# load trained model pipeline
model = joblib.load("credit_risk_model.pkl")

def risk_scoring_engine(input_data):

    # convert input dict to dataframe
    df = pd.DataFrame([input_data])

    # get the columns expected by the model
    try:
        required_cols = model.named_steps["preprocess"].feature_names_in_
    except:
        required_cols = df.columns

    # add any missing columns with default value
    for col in required_cols:
        if col not in df.columns:
            df[col] = 0

    # keep columns in correct order
    df = df[required_cols]

    # model prediction
    prob = model.predict_proba(df)[0][1]

    loan_amount = df["loanAmount"].values[0]

    expected_loss = prob * loan_amount

    if expected_loss < 50000:
        priority = "Low"
        action = "Send automated reminder"
    elif expected_loss < 200000:
        priority = "Medium"
        action = "Call borrower and discuss repayment plan"
    elif expected_loss < 500000:
        priority = "High"
        action = "Assign recovery officer"
    else:
        priority = "Critical"
        action = "Legal escalation"

    return {
        "default_probability": float(prob),
        "expected_loss": float(expected_loss),
        "priority": priority,
        "recommended_action": action
    }

In [ ]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
from recommendation_engine import risk_scoring_engine
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

app = FastAPI(title="CreditPath AI Risk Scoring API")

class BorrowerInput(BaseModel):
    loanAmount: float
    interestRate: float
    annualIncome: float
    dtiRatio: float

@app.post("/risk-score")
def get_risk_score(data: BorrowerInput):

    input_data = data.dict()

    result = risk_scoring_engine(input_data)

    return result

In [4]:
!pip install fastapi uvicorn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [fastapi]


In [ ]:
!uvicorn app:app --reload